# 04_custom_logger — Write your own logger

A logger is a *sink* for the business events that aspects emit via `box.info/warning/critical(...)`. Subclass `BaseLogger`, implement the single `write(...)` method, and the coordinator handles validation, template substitution, and fan-out. This one ships structured records to a list and subscribes to `WARNING` only — so the `INFO` line is dropped.

Companion guide: [Your own logger](../../docs/how-to/authoring-logger.md).

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio
import json
from typing import Any

from pydantic import Field

from aoa.action_machine.auth import NoneRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.logging import Channel, Level
from aoa.action_machine.logging.base_logger import BaseLogger
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## The example

In [ ]:
# ── The logger: ship each accepted line as a structured record to a sink ─────
class JsonLinesLogger(BaseLogger):
    def __init__(self, sink: list[dict[str, Any]]) -> None:
        super().__init__()                 # sets up the subscribe/match/handle pipeline
        self._sink = sink

    async def write(self, scope, message, var, ctx, state, params, indent) -> None:
        self._sink.append({
            "level": var["level"].name,            # "INFO" / "WARNING" / "CRITICAL"
            "channels": var["channels"].names,     # "business", ...
            "message": self.strip_ansi_codes(message),  # already substituted by coordinator
        })

In [ ]:
# ── A normal Action that logs business events ────────────────────────────────
class RootDomain(BaseDomain):
    name = "root"
    description = "Root domain"


class PayParams(BaseParams):
    amount: float = Field(description="Amount")


class PayResult(BaseResult):
    ok: bool = Field(description="Accepted")


@meta(description="Take a payment", domain=RootDomain)
@check_roles(NoneRole)
class PayAction(BaseAction[PayParams, PayResult]):
    @summary_aspect("Charge")
    async def charge_summary(self, params, state, box, connections):
        await box.info(Channel.business, "Charging {%var.amount}", amount=params.amount)
        await box.warning(Channel.business, "Large amount {%var.amount} flagged", amount=params.amount)
        return PayResult(ok=True)

## Run

Colab already runs an event loop, so call `await main()` at the top level instead of `asyncio.run(main())`.

In [ ]:
async def main() -> None:
    sink: list[dict[str, Any]] = []
    logger = JsonLinesLogger(sink).subscribe("warn-only", levels=Level.warning)  # filter: WARNING only
    machine = ActionProductMachine(log_coordinator=LogCoordinator(loggers=[logger]))

    await machine.run(Context(), PayAction(), PayParams(amount=5000.0), {})

    print("shipped records (INFO dropped by subscription):")
    for rec in sink:
        print("  " + json.dumps(rec, ensure_ascii=False))

await main()